# Train Embedding Model 1 — MiniLM (Vietnamese Product Search)

Notebook fine-tune **`sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`** (model 1).

Output: `embedding_project/models/minilm_finetuned_final/`

**Model 2 (BGE-M3):** dùng notebook `train_embedding_model_bge_m3.ipynb`

## 1) Cài thư viện

In [ ]:
!pip -q install sentence-transformers transformers torch datasets pandas scikit-learn numpy tqdm

## 2) Clone repo từ GitHub + kiểm tra GPU

In [ ]:
import os
import torch

# TODO: đổi URL repo của bạn
GITHUB_REPO_URL = "https://github.com/your-username/your-repo.git"
REPO_DIR = "/content/llm_provider_benchmarking"

if os.path.exists(REPO_DIR):
    !rm -rf /content/llm_provider_benchmarking

!git clone "$GITHUB_REPO_URL" "$REPO_DIR"

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 3) Thiết lập đường dẫn dữ liệu (từ repo GitHub)

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path('/content/llm_provider_benchmarking/embedding_project')
DATA_DIR = PROJECT_ROOT / 'data'
MODELS_DIR = PROJECT_ROOT / 'models'
OUTPUT_EVAL_DIR = PROJECT_ROOT / 'outputs' / 'evaluation'

MODELS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_EVAL_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATA_DIR exists:', DATA_DIR.exists())

## 4) Load dữ liệu train/valid

In [ ]:
import json
from datasets import Dataset

def load_jsonl(path):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

train_rows = load_jsonl(DATA_DIR / 'train_cleaned.jsonl')
valid_rows = load_jsonl(DATA_DIR / 'valid_cleaned.jsonl')
print('train:', len(train_rows), 'valid:', len(valid_rows))

train_ds = Dataset.from_list([{'anchor': r['query'], 'positive': r['positive']} for r in train_rows])
valid_ds = Dataset.from_list([{'anchor': r['query'], 'positive': r['positive']} for r in valid_rows])

## 5) Fine-tune model với MultipleNegativesRankingLoss (CPU preset)

In [ ]:
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers.training_args import SentenceTransformerTrainingArguments, BatchSamplers

BASE_MODEL = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
model = SentenceTransformer(BASE_MODEL)
model.max_seq_length = 256  # CPU-friendly
loss = MultipleNegativesRankingLoss(model)

args = SentenceTransformerTrainingArguments(
    output_dir=str(MODELS_DIR),
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    fp16=False,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    save_strategy='epoch',
    eval_strategy='epoch',
    logging_steps=50,
    run_name='minilm_vi_embedding_colab_cpu',
    report_to=[]
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    loss=loss,
)

trainer.train()
final_model_dir = MODELS_DIR / 'minilm_finetuned_final'
model.save(str(final_model_dir))
print('Saved model to:', final_model_dir)

## 6) Evaluate pretrained vs fine-tuned (Recall@10, Precision@10, MRR@10, NDCG@10)

In [ ]:
import numpy as np
import pandas as pd

def normalize(x):
    return x / (np.linalg.norm(x, axis=1, keepdims=True) + 1e-12)

def topk_indices(query_emb, corpus_emb, k=10):
    scores = query_emb @ corpus_emb.T
    idx = np.argpartition(-scores, kth=min(k, scores.shape[1]-1), axis=1)[:, :k]
    rows = np.arange(scores.shape[0])[:, None]
    top_scores = scores[rows, idx]
    order = np.argsort(-top_scores, axis=1)
    return idx[rows, order]

def metrics_at_k(retrieved, queries, labels, k=10):
    p_list, r_list, mrr_list, ndcg_list = [], [], [], []
    for q, hits in zip(queries, retrieved):
        rel = labels.get(q, set())
        if not rel:
            continue
        top_hits = hits[:k]
        flags = [1 if h in rel else 0 for h in top_hits]
        hit_count = sum(flags)
        p_list.append(hit_count / k)
        r_list.append(hit_count / len(rel))

        rr = 0.0
        for i, f in enumerate(flags, 1):
            if f:
                rr = 1.0 / i
                break
        mrr_list.append(rr)

        dcg = sum(f / np.log2(i + 2) for i, f in enumerate(flags))
        ideal = [1] * min(len(rel), k) + [0] * (k - min(len(rel), k))
        idcg = sum(f / np.log2(i + 2) for i, f in enumerate(ideal))
        ndcg_list.append(dcg / idcg if idcg > 0 else 0.0)

    return {
        'Precision@10': float(np.mean(p_list)) if p_list else 0.0,
        'Recall@10': float(np.mean(r_list)) if r_list else 0.0,
        'MRR@10': float(np.mean(mrr_list)) if mrr_list else 0.0,
        'NDCG@10': float(np.mean(ndcg_list)) if ndcg_list else 0.0,
    }

test_rows = load_jsonl(DATA_DIR / 'test_cleaned.jsonl')
labels_rows = json.loads((DATA_DIR / 'query_product_labels_cleaned.json').read_text(encoding='utf-8'))
labels = {r['query']: set(r['relevant_product_ids']) for r in labels_rows}

products_df = pd.read_csv(DATA_DIR / 'merged_products_vi_cleaned.csv')
products_df = products_df.dropna(subset=['product_id', 'searchable_text']).drop_duplicates('product_id')
product_ids = products_df['product_id'].astype(str).tolist()
corpus_texts = products_df['searchable_text'].astype(str).tolist()
query_texts = sorted({r['query'] for r in test_rows if r['query'] in labels})

def evaluate_model(model_name_or_path):
    m = SentenceTransformer(model_name_or_path)
    ce = normalize(m.encode(corpus_texts, batch_size=128, show_progress_bar=True, convert_to_numpy=True))
    qe = normalize(m.encode(query_texts, batch_size=128, show_progress_bar=True, convert_to_numpy=True))
    idx = topk_indices(qe, ce, k=10)
    retrieved = [[product_ids[i] for i in row] for row in idx]
    return metrics_at_k(retrieved, query_texts, labels, k=10)

pretrained_metrics = evaluate_model(BASE_MODEL)
finetuned_metrics = evaluate_model(str(final_model_dir))

result = {'pretrained': pretrained_metrics, 'finetuned': finetuned_metrics}
print(json.dumps(result, ensure_ascii=False, indent=2))
(OUTPUT_EVAL_DIR / 'metrics.json').write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')

## 7) (Optional) Kết thúc sau bước evaluate

Bạn có thể dừng notebook tại đây nếu chỉ cần train + evaluate.

In [ ]:
print('Train + evaluate hoàn tất. Không chạy bước generate embeddings/Qdrant trong notebook này.')